In [1]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)



In [2]:
df_raw = pd.read_csv('oil_sales_assignment_dataset.csv')
print("Shape:", df_raw.shape)
print("Columns:", list(df_raw.columns))
df_raw.head(3)


Shape: (2000, 13)
Columns: ['city', 'store_name', 'manufacturer', 'brand', 'class', 'size', 'sku', 'price_bracket', 'year', 'month', 'value_sales', 'volume_sales', 'average_price']


,city,store_name,manufacturer,brand,class,size,sku,price_bracket,year,month,value_sales,volume_sales,average_price
0,AL BAHA,HM No 57296 GS-CENTER-AL BAHA MAIN RD AL BAHA,NOVA FOODS,LARA,COCONUT,0.75L,LARA COCONUT 0.75L TWIN PACK,21-30,2024,12,830.86,30.1,27.6
1,AL KHARJ,HM No 55697 GS-CENTER-AL KHARJ MAIN RD AL K...,PALM & GRAIN GROUP,NAJMA,CANOLA,0.5L,NAJMA CANOLA 0.5L TWIN PACK,41-50,2024,10,373.10,9.1,41.0
2,RIYADH,HM No 86781 GS-CENTER-RIYADH MAIN RD RIYADH,AL HILAL INDUSTRIES,BAYTNA,SUNFLOWER,0.75L,BAYTNA SUNFLOWER 0.75L ECO,101+,2023,1,171.70,1.7,101.0


In [3]:
print("Target stats (volume_sales):")
print(df_raw['volume_sales'].describe().round(3))
print()
print("Null counts:", df_raw.isnull().sum().to_dict())


Target stats (volume_sales):
count    2000.000
mean        9.972
std         9.862
min         0.500
25%         2.900
50%         7.000
75%        13.800
max        81.700
Name: volume_sales, dtype: float64

Null counts: {'city': 0, 'store_name': 0, 'manufacturer': 0, 'brand': 0, 'class': 0, 'size': 0, 'sku': 0, 'price_bracket': 0, 'year': 0, 'month': 0, 'value_sales': 0, 'volume_sales': 0, 'average_price': 0}


In [4]:
df = df_raw.copy()


def parse_price_bracket(pb):
    pb = pb.strip().replace("$","")
    if pb == "101+":
        return 110.0
    parts = pb.split("-")
    return (float(parts[0]) + float(parts[1])) / 2.0

df["price_mid"] = df["price_bracket"].apply(parse_price_bracket)


cat_cols = ["city", "manufacturer", "brand", "class", "size"]
label_maps = {}
for col in cat_cols:
    unique_vals = sorted(df[col].unique())
    label_maps[col] = {v: i for i, v in enumerate(unique_vals)}
    df[col + "_enc"] = df[col].map(label_maps[col])


feature_cols = [c + "_enc" for c in cat_cols] + ["year", "month", "price_mid", "value_sales"]
target_col   = "volume_sales"

X_all = df[feature_cols].values.astype(float)
y_all = df[target_col].values.astype(float)

print(f"Features ({len(feature_cols)}): {feature_cols}")
print(f"X shape: {X_all.shape}, y shape: {y_all.shape}")
print(f"Target range: {y_all.min():.2f} — {y_all.max():.2f}, mean: {y_all.mean():.2f}")
print()

import pandas as pd
corr_df = pd.DataFrame(X_all, columns=feature_cols)
corr_df["volume_sales"] = y_all
print("Feature correlations with volume_sales:")
print(corr_df.corr()["volume_sales"].drop("volume_sales").round(4).to_string())

Features (9): ['city_enc', 'manufacturer_enc', 'brand_enc', 'class_enc', 'size_enc', 'year', 'month', 'price_mid', 'value_sales']
X shape: (2000, 9), y shape: (2000,)
Target range: 0.50 — 81.70, mean: 9.97

Feature correlations with volume_sales:
city_enc           -0.0231
manufacturer_enc    0.0266
brand_enc           0.0351
class_enc          -0.0188
size_enc            0.0080
year               -0.0084
month               0.0196
price_mid           0.0276
value_sales         0.8384


In [5]:
def train_val_test_split(X, y, train_frac=0.70, val_frac=0.15, seed=42):
    n = len(X)
    idx = np.arange(n)
    rng = np.random.default_rng(seed)
    rng.shuffle(idx)
    n_train = int(n * train_frac)
    n_val   = int(n * val_frac)
    tr = idx[:n_train]
    va = idx[n_train:n_train + n_val]
    te = idx[n_train + n_val:]
    return X[tr], X[va], X[te], y[tr], y[va], y[te]

X_tr, X_va, X_te, y_tr, y_va, y_te = train_val_test_split(X_all, y_all)
print(f"Train: {len(X_tr)}  Val: {len(X_va)}  Test: {len(X_te)}")


Train: 1400  Val: 300  Test: 300


In [6]:
def compute_mean_std(X):
    mean = X.mean(axis=0)
    std  = X.std(axis=0)
    std  = np.where(std == 0, 1.0, std)   
    return mean, std

def standardize(X, mean, std):
    return (X - mean) / std

mean_r, std_r = compute_mean_std(X_tr)
X_tr_s = standardize(X_tr, mean_r, std_r)
X_va_s = standardize(X_va, mean_r, std_r)
X_te_s = standardize(X_te, mean_r, std_r)
print("Standardization done using training stats only.")
print(f"Train mean (post-scale): {X_tr_s.mean(axis=0).round(3)}")


Standardization done using training stats only.
Train mean (post-scale): [-0.  0.  0. -0.  0.  0. -0. -0. -0.]


In [7]:
train_mean = float(y_tr.mean())
y_pred_base = np.full(len(y_te), train_mean)

def mae(y_true, y_pred):   return float(np.mean(np.abs(y_true - y_pred)))
def mse(y_true, y_pred):   return float(np.mean((y_true - y_pred)**2))
def rmse(y_true, y_pred):  return float(np.sqrt(mse(y_true, y_pred)))
def r2(y_true, y_pred):
    ss_res = np.sum((y_true - y_pred)**2)
    ss_tot = np.sum((y_true - np.mean(y_true))**2)
    return float(1 - ss_res / ss_tot) if ss_tot != 0 else 0.0

print(f"Training mean (baseline prediction): {train_mean:.4f}")
print(f"Baseline MAE  : {mae(y_te, y_pred_base):.4f}")
print(f"Baseline RMSE : {rmse(y_te, y_pred_base):.4f}")
print(f"Baseline R²   : {r2(y_te, y_pred_base):.4f}")


Training mean (baseline prediction): 9.7831
Baseline MAE  : 7.3506
Baseline RMSE : 9.7933
Baseline R²   : -0.0000


In [8]:
class LinearRegressionGD:
    def __init__(self, lr=0.01, n_iter=1000):
        self.lr, self.n_iter = lr, n_iter
        self.w = self.b = None
        self.train_loss = []
        self.val_loss   = []

    def fit(self, X_tr, y_tr, X_va=None, y_va=None):
        n, p = X_tr.shape
        self.w = np.zeros(p)
        self.b = 0.0
        for _ in range(self.n_iter):
            y_hat = X_tr @ self.w + self.b
            res   = y_tr - y_hat
            dw    = -(2/n) * (X_tr.T @ res)
            db    = -(2/n) * np.sum(res)
            self.w -= self.lr * dw
            self.b -= self.lr * db
            self.train_loss.append(float(np.mean(res**2)))
            if X_va is not None:
                val_hat = X_va @ self.w + self.b
                self.val_loss.append(float(np.mean((y_va - val_hat)**2)))

    def predict(self, X):
        return X @ self.w + self.b

model_lr = LinearRegressionGD(lr=0.05, n_iter=1000)
model_lr.fit(X_tr_s, y_tr, X_va_s, y_va)
print("Training complete.")
print(f"Final train loss (MSE): {model_lr.train_loss[-1]:.4f}")
print(f"Final val   loss (MSE): {model_lr.val_loss[-1]:.4f}")


Training complete.
Final train loss (MSE): 17.5934
Final val   loss (MSE): 18.5280


In [9]:
y_pred_reg = model_lr.predict(X_te_s)

reg_results = {
    'Baseline (mean)': dict(MAE=mae(y_te,y_pred_base), RMSE=rmse(y_te,y_pred_base), R2=r2(y_te,y_pred_base)),
    'Linear Regression': dict(MAE=mae(y_te,y_pred_reg),  RMSE=rmse(y_te,y_pred_reg),  R2=r2(y_te,y_pred_reg)),
}

print(f"{'Model':<22} {'MAE':>8} {'RMSE':>8} {'R²':>8}")
print("-"*50)
for name, m in reg_results.items():
    print(f"{name:<22} {m['MAE']:>8.4f} {m['RMSE']:>8.4f} {m['R2']:>8.4f}")


Model                       MAE     RMSE       R²
--------------------------------------------------
Baseline (mean)          7.3506   9.7933  -0.0000
Linear Regression        2.7860   4.0587   0.8282


In [10]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))


ax = axes[0]
ax.plot(model_lr.train_loss, label='Train MSE', color='#4C72B0', lw=1.5)
ax.plot(model_lr.val_loss,   label='Val MSE',   color='#DD8452', lw=1.5)
ax.set_title('Regression Loss Curves', fontweight='bold')
ax.set_xlabel('Iteration'); ax.set_ylabel('MSE')
ax.legend(); ax.grid(True, alpha=0.4)


ax = axes[1]
ax.scatter(y_te, y_pred_reg, alpha=0.4, s=15, color='#4C72B0', label='Predictions')
lims = [min(y_te.min(), y_pred_reg.min()), max(y_te.max(), y_pred_reg.max())]
ax.plot(lims, lims, 'r--', lw=1.5, label='Perfect fit')
ax.set_title('Actual vs Predicted (volume_sales)', fontweight='bold')
ax.set_xlabel('Actual'); ax.set_ylabel('Predicted')
ax.legend(); ax.grid(True, alpha=0.4)


ax = axes[2]
residuals = y_te - y_pred_reg
ax.scatter(y_pred_reg, residuals, alpha=0.4, s=15, color='#55A868')
ax.axhline(0, color='red', lw=1.5, linestyle='--')
ax.set_title('Residuals vs Predicted', fontweight='bold')
ax.set_xlabel('Predicted'); ax.set_ylabel('Residual')
ax.grid(True, alpha=0.4)

plt.tight_layout()
plt.savefig('regression_plots.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: regression_plots.png")


Saved: regression_plots.png


In [11]:
abs_err = np.abs(y_te - y_pred_reg)
top5_idx = np.argsort(abs_err)[-5:][::-1]

print("Top 5 largest absolute errors:")
print("{:>10} {:>12} {:>12}".format("Actual", "Predicted", "Abs Error"))
print("-"*38)
for i in top5_idx:
    print("{:>10.3f} {:>12.3f} {:>12.3f}".format(y_te[i], y_pred_reg[i], abs_err[i]))

print()
print("Mean abs error overall : {:.4f}".format(abs_err.mean()))
print("Errors > 2x mean       : {} rows".format((abs_err > 2*abs_err.mean()).sum()))
print()
print("Model R2 vs Baseline R2: {:.4f} vs {:.4f}".format(r2(y_te,y_pred_reg), r2(y_te,y_pred_base)))
print("Improvement in RMSE    : {:.4f} units".format(rmse(y_te,y_pred_base)-rmse(y_te,y_pred_reg)))

Top 5 largest absolute errors:
    Actual    Predicted    Abs Error
--------------------------------------
    27.200       46.459       19.259
    31.700       15.199       16.501
    35.000       19.953       15.047
    31.600       45.761       14.161
    52.600       65.730       13.130

Mean abs error overall : 2.7860
Errors > 2x mean       : 46 rows

Model R2 vs Baseline R2: 0.8282 vs -0.0000
Improvement in RMSE    : 5.7345 units


In [12]:
df_h = pd.read_csv('heart_disease_risk_2026.csv')
print("Shape:", df_h.shape)
print("Target distribution:")
print(df_h['has_heart_disease'].value_counts().to_dict())
print(f"Class balance: {df_h['has_heart_disease'].mean()*100:.1f}% positive")
df_h.head(3)


Shape: (9000, 27)
Target distribution:
{0: 6273, 1: 2727}
Class balance: 30.3% positive


,patient_id,age,sex,resting_bp_systolic,resting_bp_diastolic,cholesterol_total,hdl,ldl,triglycerides,fasting_blood_sugar,...,family_history,smoker_status,alcohol_units_per_week,exercise_minutes_per_week,sleep_hours,stress_score,wearable_owner,daily_steps,diet_quality_score,has_heart_disease
0,1,44,Male,117,74,193,57,106,119,112,...,False,Never,2.9,86,5.4,19.8,True,7731,62.9,0
1,2,57,Male,139,94,185,69,110,35,114,...,False,Never,3.0,132,4.3,45.8,True,2629,74.6,1
2,3,29,Male,128,78,197,52,108,157,95,...,False,Current,3.5,128,5.1,17.7,True,9290,65.7,0


In [13]:
df_hc = df_h.copy()


df_hc = df_hc.drop(columns=['patient_id'])


df_hc['sex_enc'] = (df_hc['sex'] == 'Male').astype(int)


cp_map = {v: i for i, v in enumerate(sorted(df_hc['chest_pain_type'].unique()))}
df_hc['chest_pain_enc'] = df_hc['chest_pain_type'].map(cp_map)


sm_map = {v: i for i, v in enumerate(sorted(df_hc['smoker_status'].unique()))}
df_hc['smoker_enc'] = df_hc['smoker_status'].map(sm_map)


bool_cols = ['exercise_induced_angina', 'family_history', 'wearable_owner']
for col in bool_cols:
    df_hc[col] = df_hc[col].astype(int)


clf_features = [
    'age', 'sex_enc', 'resting_bp_systolic', 'resting_bp_diastolic',
    'cholesterol_total', 'hdl', 'ldl', 'triglycerides',
    'fasting_blood_sugar', 'hba1c', 'bmi', 'resting_heart_rate',
    'max_heart_rate_achieved', 'chest_pain_enc', 'exercise_induced_angina',
    'st_depression', 'family_history', 'smoker_enc',
    'alcohol_units_per_week', 'exercise_minutes_per_week',
    'sleep_hours', 'stress_score', 'wearable_owner',
    'daily_steps', 'diet_quality_score'
]

X_hc = df_hc[clf_features].values.astype(float)
y_hc = df_hc['has_heart_disease'].values.astype(int)
print(f"Features ({len(clf_features)}): {clf_features[:6]}... (total {len(clf_features)})")
print(f"X shape: {X_hc.shape}  y unique: {np.unique(y_hc)}")


Features (25): ['age', 'sex_enc', 'resting_bp_systolic', 'resting_bp_diastolic', 'cholesterol_total', 'hdl']... (total 25)
X shape: (9000, 25)  y unique: [0 1]


In [14]:
X_htr, X_hva, X_hte, y_htr, y_hva, y_hte = train_val_test_split(X_hc, y_hc)
print(f"Train: {len(X_htr)}  Val: {len(X_hva)}  Test: {len(X_hte)}")
print(f"Train class balance: {y_htr.mean()*100:.1f}% positive")
print(f"Test  class balance: {y_hte.mean()*100:.1f}% positive")


Train: 6300  Val: 1350  Test: 1350
Train class balance: 29.7% positive
Test  class balance: 31.1% positive


In [15]:
mean_c, std_c = compute_mean_std(X_htr)
X_htr_s = standardize(X_htr, mean_c, std_c)
X_hva_s = standardize(X_hva, mean_c, std_c)
X_hte_s = standardize(X_hte, mean_c, std_c)
print("Standardization done using training stats only.")


Standardization done using training stats only.


In [16]:
majority_class = int(np.bincount(y_htr).argmax())
y_pred_base_clf = np.full(len(y_hte), majority_class, dtype=int)

def confusion_values(y_true, y_pred):
    TP = int(np.sum((y_true==1)&(y_pred==1)))
    FP = int(np.sum((y_true==0)&(y_pred==1)))
    FN = int(np.sum((y_true==1)&(y_pred==0)))
    TN = int(np.sum((y_true==0)&(y_pred==0)))
    return TP, FP, FN, TN

def accuracy(y_true, y_pred):  return float(np.mean(y_true==y_pred))
def precision(y_true, y_pred):
    TP,FP,FN,TN = confusion_values(y_true,y_pred)
    return TP/(TP+FP) if (TP+FP)>0 else 0.0
def recall(y_true, y_pred):
    TP,FP,FN,TN = confusion_values(y_true,y_pred)
    return TP/(TP+FN) if (TP+FN)>0 else 0.0
def f1(y_true, y_pred):
    p=precision(y_true,y_pred); r=recall(y_true,y_pred)
    return 2*p*r/(p+r) if (p+r)>0 else 0.0

print(f"Majority class: {majority_class}")
print(f"Baseline Accuracy : {accuracy(y_hte, y_pred_base_clf):.4f}")
print(f"Baseline Precision: {precision(y_hte, y_pred_base_clf):.4f}")
print(f"Baseline Recall   : {recall(y_hte, y_pred_base_clf):.4f}")
print(f"Baseline F1       : {f1(y_hte, y_pred_base_clf):.4f}")


Majority class: 0
Baseline Accuracy : 0.6889
Baseline Precision: 0.0000
Baseline Recall   : 0.0000
Baseline F1       : 0.0000


In [17]:
class LogisticRegressionGD:
    def __init__(self, lr=0.1, n_iter=500, threshold=0.5):
        self.lr, self.n_iter, self.threshold = lr, n_iter, threshold
        self.w = self.b = None
        self.train_loss = []
        self.val_loss   = []

    @staticmethod
    def _sigmoid(z):
        return 1.0 / (1.0 + np.exp(-np.clip(z, -500, 500)))

    @staticmethod
    def _bce(y, yh):
        eps = 1e-12
        yh  = np.clip(yh, eps, 1-eps)
        return float(-np.mean(y*np.log(yh) + (1-y)*np.log(1-yh)))

    def fit(self, X_tr, y_tr, X_va=None, y_va=None):
        n, p = X_tr.shape
        self.w = np.zeros(p); self.b = 0.0
        y_tr = y_tr.astype(float)
        for _ in range(self.n_iter):
            yh   = self._sigmoid(X_tr @ self.w + self.b)
            err  = yh - y_tr
            self.w -= self.lr * (1/n) * (X_tr.T @ err)
            self.b -= self.lr * (1/n) * np.sum(err)
            self.train_loss.append(self._bce(y_tr, yh))
            if X_va is not None:
                vh = self._sigmoid(X_va @ self.w + self.b)
                self.val_loss.append(self._bce(y_va.astype(float), vh))

    def predict_proba(self, X):
        return self._sigmoid(X @ self.w + self.b)

    def predict(self, X):
        return (self.predict_proba(X) >= self.threshold).astype(int)

model_clf = LogisticRegressionGD(lr=0.5, n_iter=500)
model_clf.fit(X_htr_s, y_htr, X_hva_s, y_hva)
print("Training complete.")
print(f"Final train BCE: {model_clf.train_loss[-1]:.4f}")
print(f"Final val   BCE: {model_clf.val_loss[-1]:.4f}")


Training complete.
Final train BCE: 0.2395
Final val   BCE: 0.2831


In [18]:
y_pred_clf = model_clf.predict(X_hte_s)
TP,FP,FN,TN = confusion_values(y_hte, y_pred_clf)

clf_results = {
    "Baseline (majority)" : dict(Acc=accuracy(y_hte,y_pred_base_clf),
                                  Prec=precision(y_hte,y_pred_base_clf),
                                  Rec=recall(y_hte,y_pred_base_clf),
                                  F1=f1(y_hte,y_pred_base_clf)),
    "Logistic Regression" : dict(Acc=accuracy(y_hte,y_pred_clf),
                                  Prec=precision(y_hte,y_pred_clf),
                                  Rec=recall(y_hte,y_pred_clf),
                                  F1=f1(y_hte,y_pred_clf)),
}

print("{:<24} {:>7} {:>7} {:>7} {:>7}".format("Model","Acc","Prec","Rec","F1"))
print("-"*56)
for name, m in clf_results.items():
    print("{:<24} {:>7.4f} {:>7.4f} {:>7.4f} {:>7.4f}".format(
          name, m["Acc"], m["Prec"], m["Rec"], m["F1"]))

print()
print("Confusion Matrix (test set):")
print("  TP={}  FP={}".format(TP, FP))
print("  FN={}  TN={}".format(FN, TN))

Model                        Acc    Prec     Rec      F1
--------------------------------------------------------
Baseline (majority)       0.6889  0.0000  0.0000  0.0000
Logistic Regression       0.8978  0.8730  0.7857  0.8271

Confusion Matrix (test set):
  TP=330  FP=48
  FN=90  TN=882


In [19]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))


ax = axes[0]
ax.plot(model_clf.train_loss, label="Train BCE", color="#4C72B0", lw=1.5)
ax.plot(model_clf.val_loss,   label="Val BCE",   color="#DD8452", lw=1.5)
ax.set_title("Classification Loss Curves", fontweight="bold")
ax.set_xlabel("Iteration"); ax.set_ylabel("Binary Cross-Entropy")
ax.legend(); ax.grid(True, alpha=0.4)


ax = axes[1]
cm = np.array([[TN, FP], [FN, TP]])
labels = [["TN", "FP"], ["FN", "TP"]]
im = ax.imshow(cm, cmap="Blues")
plt.colorbar(im, ax=ax)
for i in range(2):
    for j in range(2):
        ax.text(j, i, "{}\n{}".format(labels[i][j], cm[i,j]),
                ha="center", va="center", fontsize=12, fontweight="bold",
                color="white" if cm[i,j] > cm.max()*0.5 else "black")
ax.set_xticks([0,1]); ax.set_yticks([0,1])
ax.set_xticklabels(["Pred 0 (No Disease)", "Pred 1 (Disease)"])
ax.set_yticklabels(["Actual 0 (No Disease)", "Actual 1 (Disease)"])
ax.set_title("Confusion Matrix — Heart Disease", fontweight="bold")


ax = axes[2]
probas = model_clf.predict_proba(X_hte_s)
ax.hist(probas[y_hte==0], bins=40, alpha=0.6, color="#55A868", label="No Disease (0)")
ax.hist(probas[y_hte==1], bins=40, alpha=0.6, color="#C44E52", label="Disease (1)")
ax.axvline(0.5, color="black", lw=1.5, linestyle="--", label="Threshold 0.5")
ax.set_title("Predicted Probability Distribution", fontweight="bold")
ax.set_xlabel("P(has_heart_disease = 1)"); ax.set_ylabel("Count")
ax.legend(fontsize=9); ax.grid(True, alpha=0.4)

plt.tight_layout()
plt.savefig("classification_plots.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: classification_plots.png")

Saved: classification_plots.png


In [20]:
probas_te = model_clf.predict_proba(X_hte_s)


fn_mask = (y_hte == 1) & (y_pred_clf == 0)
fp_mask = (y_hte == 0) & (y_pred_clf == 1)

fn_probas = probas_te[fn_mask]
fp_probas = probas_te[fp_mask]

print(f"False Negatives (missed disease): {fn_mask.sum()}")
print(f"  Mean predicted probability for FN cases: {fn_probas.mean():.4f}")
print(f"  (These cases had low P(disease) — model was confidently wrong)")
print()
print(f"False Positives (false alarms): {fp_mask.sum()}")
print(f"  Mean predicted probability for FP cases: {fp_probas.mean():.4f}")
print()
print(f"Model F1 vs Baseline F1: {f1(y_hte,y_pred_clf):.4f} vs {f1(y_hte,y_pred_base_clf):.4f}")
print(f"Recall (catching disease cases): {recall(y_hte,y_pred_clf):.4f}")
print()


weight_df = pd.DataFrame({
    'feature': clf_features,
    'weight':  model_clf.w,
    'abs_weight': np.abs(model_clf.w)
}).sort_values('abs_weight', ascending=False)

print("Top 10 features by weight magnitude (logistic regression):")
print(weight_df.head(10)[['feature','weight']].to_string(index=False))


False Negatives (missed disease): 90
  Mean predicted probability for FN cases: 0.2386
  (These cases had low P(disease) — model was confidently wrong)

False Positives (false alarms): 48
  Mean predicted probability for FP cases: 0.6889

Model F1 vs Baseline F1: 0.8271 vs 0.0000
Recall (catching disease cases): 0.7857

Top 10 features by weight magnitude (logistic regression):
                feature    weight
max_heart_rate_achieved -2.656416
                    age -0.993002
exercise_induced_angina  0.921424
          st_depression  0.828526
                    ldl  0.439161
                    hdl -0.425290
         chest_pain_enc  0.405265
             smoker_enc -0.312123
         family_history  0.268281
     resting_heart_rate  0.231044


In [21]:
fig, ax = plt.subplots(figsize=(10, 5))
top10 = weight_df.head(10)
colors = ['#C44E52' if w > 0 else '#4C72B0' for w in top10['weight']]
ax.barh(top10['feature'], top10['weight'], color=colors, edgecolor='black', linewidth=0.5)
ax.axvline(0, color='black', lw=1)
ax.set_title('Top 10 Logistic Regression Weights (magnitude)', fontweight='bold')
ax.set_xlabel('Weight value (positive = increases risk, negative = decreases risk)')
ax.grid(True, alpha=0.4, axis='x')
plt.tight_layout()
plt.savefig('feature_weights.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: feature_weights.png")


Saved: feature_weights.png


In [22]:
print("=" * 60)
print("  FINAL RESULTS SUMMARY")
print("=" * 60)
print()
print("REGRESSION — volume_sales (oil sales dataset)")
print("-" * 60)
for name, m in reg_results.items():
    print(f"  {name:<22}  MAE={m['MAE']:.4f}  RMSE={m['RMSE']:.4f}  R²={m['R2']:.4f}")
print()
print("CLASSIFICATION — has_heart_disease (heart disease dataset)")
print("-" * 60)
for name, m in clf_results.items():
    print(f"  {name:<24}  Acc={m['Acc']:.4f}  Prec={m['Prec']:.4f}  Rec={m['Rec']:.4f}  F1={m['F1']:.4f}")
print()
print("No scikit-learn or ML library was used anywhere.")
print("All models, metrics, splits, and scaling: from scratch.")
print("=" * 60)


  FINAL RESULTS SUMMARY

REGRESSION — volume_sales (oil sales dataset)
------------------------------------------------------------
  Baseline (mean)         MAE=7.3506  RMSE=9.7933  R²=-0.0000
  Linear Regression       MAE=2.7860  RMSE=4.0587  R²=0.8282

CLASSIFICATION — has_heart_disease (heart disease dataset)
------------------------------------------------------------
  Baseline (majority)       Acc=0.6889  Prec=0.0000  Rec=0.0000  F1=0.0000
  Logistic Regression       Acc=0.8978  Prec=0.8730  Rec=0.7857  F1=0.8271

No scikit-learn or ML library was used anywhere.
All models, metrics, splits, and scaling: from scratch.
